<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/3_wb_qwin_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installs: Working on colab run time > 2026.04

In [ ]:
# 1. Remove conflicting packages
!pip uninstall -y torch torchvision torchaudio torchtext \
  triton bitsandbytes \
  transformers tokenizers accelerate peft trl datasets \
  numpy pandas pyarrow

In [ ]:
# 2. Install PyTorch CUDA 11.8 stack
# Moving from torch 2.2.0 -> 2.3.1 is safer on modern Colab / Python 3.12.
# PyTorch publishes a CUDA 11.8 wheel for torch 2.3.1.
!pip install --no-cache-dir \
  torch==2.3.1 \
  torchvision==0.18.1 \
  torchaudio==2.3.1 \
  --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# 3. Install Hugging Face + data stack
# bitsandbytes 0.45.4 avoids the old triton.ops import issue better than 0.43.x.
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  pyarrow==15.0.2 \
  transformers==4.38.2 \
  tokenizers==0.15.2 \
  peft==0.9.0 \
  accelerate==0.27.2 \
  trl==0.8.1 \
  datasets==2.18.0 \
  bitsandbytes==0.45.4 \
  scipy==1.12.0 \
  fsspec==2024.2.0

# <font color = red> RESTART RUNTIME HERE

# Imports

Your job is now to wrap wandb into this code. Right now, we have a directory structure which looks like this:
{base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/

This split_data directory has a file called split_data.parquet. In this file, we have three columns you'll care about: "input_data", "ground_truth_data", and "tts_label". the last column, tts_label, has labes of either train,validation, test, or OOT.

You should have functions which do separate parts of the process needed to fine tune.

After these organized functions you create, we should have code which assigns variables which set the base_dir, project name, and project number. Fine tuning end to end should be do-able by calling these organized functions.

Each experiment we run in wandb should have the final checkpoints etc needed saved into this {base_dir}/projects/{project_name}/project_{1,2,3,etc]}/split_data/

In [ ]:
import torch
import tokenizers
import peft
import accelerate
import trl
from datasets import Dataset
import bitsandbytes as bnb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm
from google.colab import drive
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoTokenizer
)
from sklearn.model_selection import train_test_split
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from trl import SFTTrainer
import gc
import wandb
import os

## Weights & Biases (W&B) and Project Setup

We will integrate Weights & Biases (W&B) to track experiments, log metrics, and save model artifacts. The project structure will be organized as follows:

`{base_dir}/{projects}/{project_name}/{project_[1,2,3,etc]}/split_data/`

Each W&B experiment will save its final checkpoints and other necessary artifacts within this structure.

## Fine-tuning Functions

Now, let's refactor the fine-tuning process into modular functions to improve readability, reusability, and integration with W&B.

In [ ]:
def load_and_prepare_data(data_file_path, tokenizer):
    """Loads data from a parquet file, formats it using a chat template, and returns Hugging Face Datasets for training and evaluation."""
    print(f"Loading data from {data_file_path}...")
    full_df = pd.read_parquet(data_file_path)

    # Define the formatting function for the chat template
    def format_chat_template(row):
        product_types = [
            'Checking or savings account',
            'Money transfer, virtual currency, or money service',
            'Vehicle loan or lease',
            'Credit card',
            'Payday loan, title loan, personal loan, or advance loan',
            'Credit reporting or other personal consumer reports',
            'Mortgage',
            'Student loan',
            'Debt collection',
            'Debt or credit management'
        ]
        messages = [
            {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
            {"role": "user", "content": f"Complaint: {row['input_data']}\nPredefined Product Categories: {product_types}"},
            {"role": "assistant", "content": row['ground_truth_data']}
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

    # Convert DataFrames to Hugging Face Datasets
    train_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'train']).map(format_chat_template, remove_columns=list(full_df.columns))
    eval_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'validation']).map(format_chat_template, remove_columns=list(full_df.columns))
    test_dataset = Dataset.from_pandas(full_df[full_df['tts_label'] == 'test']).map(format_chat_template, remove_columns=list(full_df.columns))

    print("Training data sample:")
    print(train_dataset[0]['text'])
    print("\nEvaluation data sample:")
    print(eval_dataset[0]['text'])

    return train_dataset, eval_dataset, test_dataset

In [ ]:
def load_qwen_model(model_name):
    """Loads the Qwen model with 4-bit quantization configuration."""
    print(f"Loading Qwen model: {model_name} with 4-bit quantization...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1 # Recommended for Qwen
    return model

In [ ]:
def setup_lora_config(model):
    """Prepares the model for k-bit training and applies LoRA configuration."""
    print("Setting up LoRA configuration...")
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16, # LoRA attention dimension
        lora_alpha=32, # Alpha parameter for LoRA scaling
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all linear layers in Qwen
        lora_dropout=0.05, # Dropout probability for LoRA layers
        bias="none", # No bias in LoRA layers
        task_type="CAUSAL_LM", # Causal Language Modeling for instruction tuning
    )

    model = get_peft_model(model, lora_config)
    print(model.print_trainable_parameters())
    return model, lora_config

In [ ]:
def configure_wandb_training_args(output_dir, run_name, num_train_epochs=3):
    """Configures TrainingArguments for SFTTrainer with W&B integration."""
    print(f"Configuring training arguments for W&B run: {run_name}...")
    training_args = TrainingArguments(
        output_dir=output_dir, # Output directory for checkpoints and logs
        num_train_epochs=num_train_epochs, # Number of training epochs
        per_device_train_batch_size=1, # Batch size per GPU/CPU for training
        gradient_accumulation_steps=4, # Number of updates steps to accumulate gradients
        gradient_checkpointing=True, # Enable gradient checkpointing for memory efficiency
        optim="paged_adamw_8bit", # Optimizer (8-bit AdamW for memory efficiency)
        logging_steps=10, # Log training metrics every N steps
        learning_rate=2e-4, # Learning rate
        fp16 = True,
        bf16=False, # Use bfloat16 if available
        tf32=False, # Use tf32 if available
        max_grad_norm=0.3, # Max gradient norm
        warmup_ratio=0.03, # Warmup ratio for learning rate scheduler
        lr_scheduler_type="constant", # Learning rate scheduler type
        report_to="wandb", # Report to W&B
        run_name=run_name, # W&B run name
        # Evaluation arguments
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
    return training_args

In [ ]:
def run_fine_tuning(model, train_dataset, eval_dataset, lora_config, tokenizer, training_args, max_seq_length=512):
    """Initializes and runs the SFTTrainer."""
    print("\nStarting Qwen fine-tuning...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
        args=training_args,
        max_seq_length=max_seq_length, # Maximum sequence length for inputs
        dataset_text_field="text", # Field in dataset containing the formatted text
        packing=True, # Pack multiple examples into one sequence for efficiency
    )

    trainer.train()
    print("\nFine-tuning complete!")

    # Save the adapter in the specified output directory
    adapter_output_dir = os.path.join(training_args.output_dir, "final_adapter")
    trainer.save_model(adapter_output_dir)
    print(f"Final adapter saved to: {adapter_output_dir}")

    return trainer

In [ ]:
def evaluate_finetuned_model(model_name, adapter_path, tokenizer, eval_df, product_types):
    """Loads the base model, merges with adapter, and evaluates classification performance."""
    print("\nEvaluating fine-tuned model...")
    # Clear VRAM and local memory before reloading model for evaluation
    gc.collect()
    torch.cuda.empty_cache()

    print("Loading base model and merging adapters... this may take a moment.")

    # Reload the base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        return_dict=True,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    # Load the PEFT model and merge
    finetuned_model = PeftModel.from_pretrained(
        base_model,
        adapter_path
    )

    print("Merging adapters...")
    finetuned_model = finetuned_model.merge_and_unload()
    finetuned_model.eval()

    print("Fine-tuned model loaded and adapters merged successfully.")

    # Update the classification function to use the fine-tuned model
    def classify_product_with_finetuned_qwen(
        complaint_row,
        model,
        tokenizer,
        product_types
        ):
        messages = [
            {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
            {"role": "user", "content": f"Complaint: {complaint_row['input_data']}\nPredefined Product Categories: {product_types}"}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        try:
            generated_ids = model.generate(
                model_inputs.input_ids,
                max_new_tokens=50,
                do_sample=False,
                temperature=0.0,
            )
            input_len = model_inputs.input_ids.shape[1]
            response_text = tokenizer.decode(generated_ids[0, input_len:], skip_special_tokens=True).strip()

            predicted_category = "Unknown"
            for p_type in product_types:
                if p_type.lower() in response_text.lower():
                    predicted_category = p_type
                    break
            if predicted_category == "Unknown" and len(response_text.split()) < 5:
                 predicted_category = response_text

            return predicted_category
        except Exception as e:
            print(f"Error classifying complaint: {e}")
            return "Error_Classification"

    print("Running fine-tuned Qwen classification on the evaluation subset (first 100 rows)...")
    # Take first 100 rows of the evaluation DataFrame (assuming eval_df contains the original data)
    eval_subset_df = pd.DataFrame(eval_df).head(100).copy() # Convert Dataset to DataFrame first

    # Apply the fine-tuned Qwen classification function
    finetuned_qwen_inf_results = eval_subset_df.apply(
        lambda row: classify_product_with_finetuned_qwen(row, finetuned_model, tokenizer, product_types),
        axis=1
    )

    eval_subset_df['finetuned_qwen_classified_product'] = finetuned_qwen_inf_results

    print("\nFine-tuned Qwen Classified Product Value Counts:")
    display(eval_subset_df['finetuned_qwen_classified_product'].value_counts())

    print("\nComparison of Original vs. Fine-tuned Qwen Classified Products (first 10 rows):")
    display(eval_subset_df[['ground_truth_data', 'finetuned_qwen_classified_product', 'input_data']].head(10))

    correct_finetuned_qwen_classifications = (eval_subset_df['ground_truth_data'] == eval_subset_df['finetuned_qwen_classified_product']).sum()
    total_finetuned_qwen_classifications = len(eval_subset_df)
    accuracy_finetuned_qwen = correct_finetuned_qwen_classifications / total_finetuned_qwen_classifications
    print(f"\nFine-tuned Qwen Classification Accuracy (vs. original 'ground_truth_data' column): {accuracy_finetuned_qwen:.2f}")

    return accuracy_finetuned_qwen, eval_subset_df

# Main Execution Section

In [ ]:
# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

# Define the model name globally
qwen_model_name = "Qwen/Qwen1.5-4B-Chat"

# Load tokenizer globally and configure padding
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token # Qwen uses EOS as padding token by default
tokenizer.padding_side = "right" # Or "left" depending on preference for inference

# Define base directory and project specific details
base_dir = "/content/drive/MyDrive"
project_name = "CFG_complaints_finetuning"
project_number = 2 # Increment this for each new experiment or set of runs

# Construct paths
project_path = os.path.join(base_dir, "projects", project_name, f"project_{project_number}")
split_data_path = os.path.join(project_path, "split_data", "split_data.parquet")


In [ ]:
### need to ensure different run number each time

In [ ]:
# Main execution block for fine-tuning

# 1. Initialize W&B
run_name = f"qwen_finetune_project_{project_number}"
wandb.init(
    project=project_name,
    name=run_name,
    config={
      "model_name": qwen_model_name,
      "lora_r": 16,
      "lora_alpha": 32,
      "num_epochs": 3,
      "learning_rate": 2e-4,
      "max_seq_length": 512
  }
)

# 2. Load and prepare data (Hugging Face Datasets format)
train_dataset, eval_dataset, test_dataset_for_eval = load_and_prepare_data(split_data_path, tokenizer)

# 3. Load Qwen model
model = load_qwen_model(qwen_model_name)

# 4. Setup LoRA
model, lora_config = setup_lora_config(model)

# 5. Configure Training Arguments
# Training output directory will be inside the project structure
training_output_dir = os.path.join(project_path, "training_output")
os.makedirs(training_output_dir, exist_ok=True)
training_args = configure_wandb_training_args(training_output_dir, run_name)

# 6. Run Fine-tuning
trainer = run_fine_tuning(model, train_dataset, eval_dataset, lora_config, tokenizer, training_args)

# 7. Evaluate the fine-tuned model
final_adapter_path = os.path.join(training_output_dir, "final_adapter")
product_types = [
    'Checking or savings account',
    'Money transfer, virtual currency, or money service',
    'Vehicle loan or lease',
    'Credit card',
    'Payday loan, title loan, personal loan, or advance loan',
    'Credit reporting or other personal consumer reports',
    'Mortgage',
    'Student loan',
    'Debt collection',
    'Debt or credit management'
]

accuracy, eval_results_df = evaluate_finetuned_model(qwen_model_name, final_adapter_path, tokenizer, test_dataset_for_eval, product_types)

# Log final accuracy to W&B
wandb.log({"final_evaluation_accuracy": accuracy})

# Save the evaluation results DataFrame as a W&B artifact
artifact = wandb.Artifact("evaluation_results", type="dataset")
with artifact.new_file("eval_results.csv", "w") as f:
    eval_results_df.to_csv(f, index=False)
wandb.log_artifact(artifact)

print("\nFine-tuning and evaluation complete. Check W&B for experiment details.")

# 8. Finish W&B run
wandb.finish()

In [ ]:
# 1. Clear VRAM and local memory
if 'model' in locals(): del model
if 'trainer' in locals(): del trainer
gc.collect()
torch.cuda.empty_cache()